In [0]:
# Import all functions and types from pyspark.sql for DataFrame transformations and schema definitions
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Create Flag Parameter

In [0]:
# Create a Databricks widget for user input to control incremental processing (default is '0')
dbutils.widgets.text('incremental_flag','0')
# Retrieve the value of the incremental_flag widget for use in conditional logic
incremental_flag = dbutils.widgets.get('incremental_flag')

#  Creating Dimensional Model

### Fetch Relative Columns

In [0]:
# Query distinct Dealer_ID and associated DealerName from the Silver layer parquet file
df_src = spark.sql('''
    select DISTINCT(Dealer_ID) as Dealer_ID, DealerName     
    from parquet.`abfss://silver@salesprojectmgr.dfs.core.windows.net`
''')

# Display the resulting DataFrame in Databricks
display(df_src)

### dim_model Sink = Initial and Incremental (for intitial load just bring the schema, for incremental load bring whole table)

In [0]:
# Check if the target dimension table exists in the catalog
if spark.catalog.tableExists('sales_catalog.gold.dim_dealer') :

    # If the table exists, load existing dimension records from the Silver layer
    df_sink = spark.sql( '''
    SELECT dim_dealer_key, Dealer_ID, DealerName
    from sales_catalog.gold.dim_dealer
    ''')

else :

    # If the table does not exist, create an empty DataFrame with the required schema
    df_sink = spark.sql( '''
    SELECT 1 as dim_dealer_key, Dealer_ID, DealerName
    from parquet.`abfss://silver@salesprojectmgr.dfs.core.windows.net`
    where 1=0
    ''')

In [0]:
display(df_sink)

### Filtering New Records and Old Records

In [0]:
# Join new distinct models from Silver with existing dimension records to identify new and existing entries
df_filter = df_src.join(df_sink,df_src['Dealer_ID'] == df_sink['Dealer_ID'],'left').select(df_src['Dealer_ID'],df_src['DealerName'],df_sink['dim_dealer_key'])

In [0]:
display(df_filter)

In [0]:
# Filter records that already exist in the dimension table (i.e., have a non-null surrogate key)
df_filter_old = df_filter.filter(col('dim_dealer_key').isNotNull())
display(df_filter_old)

In [0]:
# Select records that do not exist in the dimension table (i.e., new models with null surrogate key)
df_filter_new = df_filter.filter(col('dim_dealer_key').isNull())
display(df_filter_new)

In [0]:
# Select only the relevant columns for new dealer records: Dealer_ID and DealerName
df_filter_new = df_filter_new.select('Dealer_ID', 'DealerName')

### Create Surrogate Key

In [0]:
# Fetch the maximum surrogate key value from the dimension table based on the incremental flag
if (incremental_flag == '0'):
    max_value = 1  # initial load
else:
    # Incremental load: get current max surrogate key from the dimension table
    max_value_df = spark.sql("select max(dim_dealer_key) from sales_catalog.gold.dim_dealer")
    max_value = max_value_df.collect()[0][0] + 1

In [0]:
# Create Srrogate key column and assign a new surrogate key to each new record by adding a unique monotonically increasing ID to the current max surrogate key
df_filter_new = df_filter_new.withColumn('dim_dealer_key', max_value + monotonically_increasing_id())
display(df_filter_new)

### Final DF = df_filter_old + df_filter_new

In [0]:
# Combine new and existing records into a single DataFrame for the final dimension table
df_final = df_filter_new.union(df_filter_old)
display(df_final)

# SCD Type 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
# Incremental RUN
if spark.catalog.tableExists('sales_catalog.gold.dim_dealer'):
    delta_table = DeltaTable.forPath(spark, "abfss://gold@salesprojectmgr.dfs.core.windows.net/dim_dealer")

    delta_table.alias("trg").merge(df_final.alias("src"), "trg.dim_dealer_key = src.dim_dealer_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

# Initial RUN
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path", "abfss://gold@salesprojectmgr.dfs.core.windows.net/dim_dealer")\
        .saveAsTable("sales_catalog.gold.dim_dealer")


In [0]:
%sql
SELECT * FROM sales_catalog.gold.dim_dealer